# SVG2 stages 3–4 by hand: cleanup + DAM descriptions for every object trajectory

**Input: the SAM2 output we already saved.** `data/stage12/sav_000001_first120_tracks.json` holds 63 objects × 120 frames of masks, made by `notebooks/sam2_video_step_by_step.ipynb`. **SAM2 is not run again.** The only other input is the video file, which ships with their code.

| stage | what | code (`third_party/SVG2/pipeline/svg2_pipeline.py`) | paper |
|---|---|---|---|
| 3 cleanup | remove duplicate / part tracks, smooth masks | `stage3_cleanup` l.1530, `dedupe_tracks` l.595 | p. 23 |
| 4 caption | the 40 largest objects → 8 frames each → DAM-3B-Video → one paragraph per object | `stage4_caption` l.1558, `DamCaptioner` l.779 | p. 24 (A.2) |

**Output:**
- every object's paragraph, shown next to its 8 frames, with the exact prompt highlighted;
- a look inside DAM for one object: the full view, the focal crop, tensor shapes, and how many image tokens go into the language model;
- a randomness check: the same object described several times;
- saved to `data/stage4/` (descriptions JSON) and `notes/stage4/` (pictures + README). Push both.

**Time on a T4:** a few minutes of setup and model download (~6 GB), then roughly 10–40 s per object. With `N_DESCRIBE = 40` expect about 10–25 min. Lower it for a quick try.

In [ ]:
# ---- setup (same as the SAM2 notebook) + DAM, installed exactly as the SVG2 README says ----
import os, subprocess, sys, time, json
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # one GPU: DAM loads with device_map="auto"; keep it on a single T4
REPO = "/kaggle/working/svg2-implementation"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-b", "claude/traser-implementation-guide-xlshit",
                    "https://github.com/kanu2704/svg2-implementation", REPO], check=True)
os.chdir(REPO)
os.environ["HF_HOME"] = "/tmp/hf_cache"
r = subprocess.run("bash setup_kaggle.sh", shell=True, capture_output=True, text=True)
print(r.stdout[-800:], r.stderr[-800:])
# SVG2 README, install step 3: "keep --no-deps, its dependency list would replace transformers"
r = subprocess.run('python3 -c "import dam" 2>/dev/null || python3 -m pip install -q --no-deps '
                   '"git+https://github.com/NVlabs/describe-anything.git"; python3 -m pip install -q sentencepiece',
                   shell=True, capture_output=True, text=True)
print(r.stdout[-800:], r.stderr[-800:])
sys.path.insert(0, os.path.join(REPO, "third_party/SVG2/pipeline"))

In [ ]:
import html, difflib, colorsys
import numpy as np, torch, cv2
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from PIL import Image
import svg2_pipeline as P
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "|", dev, torch.cuda.get_device_name(0) if dev == "cuda" else "(no GPU: turn it on!)")

TRACKS = "data/stage12/sav_000001_first120_tracks.json"     # saved by the SAM2 notebook
VIDEO = "third_party/SVG2/pipeline/examples/sav_000001.mp4"
for p in (TRACKS, VIDEO):
    print(("OK      " if os.path.exists(p) else "MISSING ") + p)
assert os.path.exists(TRACKS), "run `git pull` first: the tracks file comes from the SAM2 notebook"

## Settings

These are their defaults from `CaptionConfig` (`svg2_pipeline.py:223-250`), shown so you can see every knob.

In [ ]:
cfg = P.PipelineConfig()                       # their defaults
N_DESCRIBE = cfg.caption.max_objects           # 40; lower for a quick try (e.g. 5)
for k, v in vars(cfg.caption).items():
    print(f"  caption.{k} = {v!r}")
for k, v in vars(cfg.cleanup).items():
    print(f"  cleanup.{k} = {v!r}")
OUT, DATA = "notes/stage4", "data/stage4"
os.makedirs(OUT, exist_ok=True); os.makedirs(DATA, exist_ok=True)
LOG = []
def say(*a):
    s = " ".join(str(x) for x in a); print(s); LOG.append(s)

def colour(i):
    return np.array(colorsys.hsv_to_rgb((i * 0.618034) % 1.0, 0.75, 1.0))

def overlay(frame, masks, alpha=0.5, label=True):
    img = frame.astype(np.float32) / 255
    for i, m in masks.items():
        img[m] = (1 - alpha) * img[m] + alpha * colour(i)
    img = (img * 255).astype(np.uint8)
    if label:
        for i, m in masks.items():
            ys, xs = np.nonzero(m)
            if len(xs):
                p = (int(xs.mean()) - 6, int(ys.mean()) + 5)
                cv2.putText(img, str(i), p, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 3)
                cv2.putText(img, str(i), p, cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    return img

def show(images, titles=None, cols=4, size=3.2, save=None, suptitle=None, dpi=60):
    rows = int(np.ceil(len(images) / cols))
    h, w = images[0].shape[:2]
    fig, axes = plt.subplots(rows, cols, figsize=(cols * size, rows * size * h / w + 0.5), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for k, im in enumerate(images):
        axes.ravel()[k].imshow(im, cmap="gray" if im.ndim == 2 else None)
        if titles:
            axes.ravel()[k].set_title(titles[k], fontsize=9)
    if suptitle:
        fig.suptitle(suptitle)
    fig.tight_layout()
    if save:
        fig.savefig(f"{OUT}/{save}", dpi=dpi)
    plt.show()

## 1. Load the saved SAM2 tracks and the video frames

Each object is an id plus 120 masks stored compressed (COCO RLE). `None` means "not visible in that frame". The frames come straight from the mp4, as in their pipeline (`read_video_frames`).

In [ ]:
tracks = json.load(open(TRACKS))
T, H, W = tracks["total_frames"], tracks["height"], tracks["width"]
frames_all, fps, _, _ = P.read_video_frames(VIDEO)
frames = frames_all[:T]; del frames_all
visible = {o["object_id"]: sum(r is not None for r in o["masks"]) for o in tracks["objects"]}
say(f"Loaded `{TRACKS}`: {len(tracks['objects'])} objects × {T} frames ({W}x{H}, {fps:.0f} fps); "
    f"{sum(visible.values())} non-empty masks. No SAM2 needed.")
idx = [0, 60, 119]
show([overlay(frames[f], {o["object_id"]: P.decode_rle(o["masks"][f]) for o in tracks["objects"] if o["masks"][f]}) for f in idx],
     [f"frame {f}" for f in idx], cols=3, size=4, suptitle="Input: the stage-2 trajectories (one colour per id)")

## 2. Stage 3 cleanup (paper p. 23; code `stage3_cleanup`)

**2a. Remove duplicate / part tracks** (`dedupe_tracks`, l.595):
- Sort the tracks by how many frames they're visible in.
- For each pair (a, b) that is visible together in at least **5** frames, compute per frame:
  - `ca` = how much of **a** lies inside b;
  - `cb` = how much of **b** lies inside a.
- `strong_a` = the share of those frames where `ca ≥ 0.9`; same for `strong_b`.
- If only `strong_a ≥ 0.7`, delete **a**, because it's a part of b. If only `strong_b ≥ 0.7`, delete **b**.
- If both are ≥ 0.7, they're the same object twice: keep the one with the higher "stability score" `0.5·frames + 0.05·median area + 200·recent IoU`.

The function below is their loop written out so it also records **why** each track was deleted. Then we check it gives exactly their result.

In [ ]:
def dedupe_explained(tracks_list, ids, c=cfg.cleanup):
    n = len(tracks_list); alive = [True] * n; why = {}
    scores = [P.track_stability_score(t) for t in tracks_list]
    order = sorted(range(n), key=lambda k: -P._track_length(tracks_list[k]))
    for i, a in enumerate(order):
        if not alive[a]:
            continue
        for b in order[i + 1:]:
            if not alive[b]:
                continue
            per = P._containment_per_frame(tracks_list[a], tracks_list[b])
            if len(per) < c.min_overlap_frames:
                continue
            sa = sum(ca >= c.contain_ratio_thresh for ca, _ in per) / len(per)
            sb = sum(cb >= c.contain_ratio_thresh for _, cb in per) / len(per)
            kill_a = kill_b = False
            if sa >= c.coverage_share_thresh and sb < c.coverage_share_thresh: kill_a = True
            elif sb >= c.coverage_share_thresh and sa < c.coverage_share_thresh: kill_b = True
            elif sa >= c.coverage_share_thresh and sb >= c.coverage_share_thresh:
                kill_b = scores[a] >= scores[b]; kill_a = not kill_b
            if kill_a:
                alive[a] = False
                why[ids[a]] = dict(kept=ids[b], shared_frames=len(per), inside_share=round(sa, 2), other_way=round(sb, 2),
                                   score_deleted=round(scores[a]), score_kept=round(scores[b]))
                break
            if kill_b:
                alive[b] = False
                why[ids[b]] = dict(kept=ids[a], shared_frames=len(per), inside_share=round(sb, 2), other_way=round(sa, 2),
                                   score_deleted=round(scores[b]), score_kept=round(scores[a]))
    return alive, why

ids = [o["object_id"] for o in tracks["objects"]]
alive, why = dedupe_explained([o["masks"] for o in tracks["objects"]], ids)
theirs = P.dedupe_tracks([o["masks"] for o in tracks["objects"]], cfg.cleanup.contain_ratio_thresh,
                         cfg.cleanup.coverage_share_thresh, cfg.cleanup.min_overlap_frames)
assert alive == theirs, "differs from their dedupe_tracks!"
for gone, w in why.items():
    say(f"Stage 3 dedupe: id {gone} deleted, kept id {w['kept']}. Visible together in {w['shared_frames']} frames; "
        f"id {gone} was ≥90% inside id {w['kept']} in {w['inside_share']:.0%} of them "
        f"(the other way round: {w['other_way']:.0%}); stability scores deleted {w['score_deleted']} vs kept {w['score_kept']}")
cleaned = P.stage3_cleanup(cfg, tracks)                         # THEIR stage 3 (dedupe + smoothing)
say(f"Stage 3: {len(tracks['objects'])} → {len(cleaned['objects'])} objects")

In [ ]:
# show each deleted track next to the one that "contains" it, in the frame where the deleted one is largest
pairs, titles = [], []
by_id = {o["object_id"]: o for o in tracks["objects"]}
for gone, w in why.items():
    f = max(range(T), key=lambda k: P.rle_area(by_id[gone]["masks"][k]))      # frame where the deleted track is largest
    m_gone = P.decode_rle(by_id[gone]["masks"][f])
    kept_rle = by_id[w["kept"]]["masks"][f]
    m_kept = P.decode_rle(kept_rle) if kept_rle else np.zeros_like(m_gone)
    pairs += [overlay(frames[f], {gone: m_gone}), overlay(frames[f], {w["kept"]: m_kept})]
    titles += [f"deleted id {gone} (frame {f})", f"kept id {w['kept']} (same frame)"]
if pairs:
    show(pairs, titles, cols=4, size=3, save="2_deleted_tracks.png",
         suptitle="Stage 3: deleted tracks (left) and the track that contains them (right)")

**2b. Smoothing** (`morphological_open`, l.639): in every frame, each mask is eroded by 1 px, then dilated by 1 px (3×3). A per-frame mask under **32 px** afterwards is deleted, and an object with no masks left disappears.

In [ ]:
rows = []
clean_by_id = {o["object_id"]: o for o in cleaned["objects"]}
for oid, o in clean_by_id.items():
    before = sum(P.rle_area(r) for r in by_id[oid]["masks"])
    after = sum(P.rle_area(r) for r in o["masks"])
    lost_frames = sum(1 for a, b in zip(by_id[oid]["masks"], o["masks"]) if a and not b)
    rows.append((oid, before, after, lost_frames))
rows.sort(key=lambda r: -(r[1] - r[2]) / max(r[1], 1))
say("Smoothing: share of pixels removed per object (top 5): " +
    ", ".join(f"id {oid} {100 * (b - a) / b:.1f}% ({lf} frames emptied)" for oid, b, a, lf in rows[:5]))

## 3. Stage 4, part 1: which objects get described (only the 40 largest)

Total area = an object's mask pixels added up over **all frames**. Keep the top `max_objects = 40` (`largest_objects`, l.1338). **This limit is not in the paper.**

In [ ]:
traj = P.ObjectTrajectories(cleaned)
targets = traj.largest_objects(N_DESCRIBE)
ranked = sorted(((traj.total_area(o), o) for o in traj.object_ids() if traj.total_area(o) > 0), reverse=True)
rank_of = {o: k + 1 for k, (_, o) in enumerate(ranked)}
say(f"Objects after stage 3: {len(ranked)}; described: {len(targets)}; dropped (no description → no name): "
    f"{[o for _, o in ranked if o not in targets]}")
fig, ax = plt.subplots(figsize=(13, 3))
ax.bar(range(len(ranked)), [a for a, _ in ranked], color=["tab:blue" if o in targets else "lightgray" for _, o in ranked])
ax.set_yscale("log"); ax.set_xticks(range(len(ranked))); ax.set_xticklabels([str(o) for _, o in ranked], fontsize=7, rotation=90)
ax.set_xlabel("object id (ranked)"); ax.set_ylabel("total mask area (px, log)")
ax.set_title(f"Blue = the {len(targets)} objects DAM describes; grey = dropped")
fig.savefig(f"{OUT}/3_ranking.png", dpi=60, bbox_inches="tight"); plt.show()

## 4. Stage 4, part 2: which 8 frames per object

**Code:** `frame_sampling = "uniform"` spreads 8 picks evenly over the frames where the object is visible (`sample_frames`, l.1344).
**Paper (p. 24):** "top 8 frames with the largest visible regions". The code has that as `"max_area"`, but it isn't the default. Both are shown for the first 5 objects.

In [ ]:
sampled = {oid: traj.sample_frames(oid, cfg.caption.frames_per_object, cfg.caption.frame_sampling) for oid in targets}
for oid in targets[:5]:
    alt = traj.sample_frames(oid, cfg.caption.frames_per_object, "max_area")
    say(f"id {oid}: visible in {len(traj._active[oid])} frames; code (uniform) → {sampled[oid]}; paper's rule (max area) → {alt}")

## 5. Load DAM and look at the exact prompt

`DamCaptioner` (their wrapper, l.779) loads `nvidia/DAM-3B-Video` in focal-prompt mode (`full+focal_crop`). The question comes from `CaptionConfig.prompt` with one `<image>` per frame. DAM wraps it in its chat template (`conv_mode="v1"`), and that full text is what the language model reads.

Colour key:
- **grey:** the system sentence;
- **blue:** speaker tags;
- **orange:** `<image>` placeholders, each replaced by one frame's image vectors;
- **green:** the instruction.

In [ ]:
t0 = time.time()
captioner = P.DamCaptioner(cfg.caption, dev)
dam_model = captioner.model                       # dam.DescribeAnythingModel
say(f"DAM loaded ({cfg.caption.model_id}, prompt mode {captioner.model.prompt_mode}) in {time.time() - t0:.0f} s")

n_img = cfg.caption.frames_per_object
query = cfg.caption.prompt.format(image_tokens=" ".join(["<image>"] * n_img))
PROMPT, _ = dam_model.get_prompt(query)           # exactly what get_description builds
INSTRUCTION = cfg.caption.prompt.split("\n", 1)[1]

def highlight(prompt, frame_ids=None, per_image=None, answer=None):
    s = html.escape(prompt)
    u = s.index("USER:")
    out = f'<span style="color:#8a8a8a">{s[:u]}</span>' + s[u:]
    out = out.replace("USER:", '<b style="color:#1f6feb">USER:</b>').replace("ASSISTANT:", '<b style="color:#1f6feb">ASSISTANT:</b>')
    out = out.replace(html.escape(INSTRUCTION), f'<span style="background:#d7f5dd;color:#0b4d1a">{html.escape(INSTRUCTION)}</span>')
    parts = out.split("&lt;image&gt;"); out = parts[0]
    for k, rest in enumerate(parts[1:]):
        lab = "&lt;image&gt;" + (f" f{frame_ids[k]}" if frame_ids else "") + (f" → {per_image} vectors" if per_image else "")
        out += f'<span style="background:#ffd28a;color:#000;padding:0 3px;border-radius:3px">{lab}</span>' + rest
    if answer is not None:
        out += f' <span style="background:#e8e0ff;color:#2b1a66">{html.escape(answer)}</span>'
    return f'<pre style="white-space:pre-wrap;font-size:12px;line-height:1.5;border:1px solid #ccc;padding:8px">{out}</pre>'

print(PROMPT)                                     # plain text, exactly as built
display(HTML(highlight(PROMPT)))

## 6. Inside DAM for one object: inputs, shapes, tokens

`DEMO_ID = 1` is the boy in our run (the largest non-floor object). For each of his 8 frames, DAM builds:
- the **full view** (whole frame + mask);
- the **focal view** (a crop of 3× the mask's box, at least 48×48 px, + mask).

Each becomes 384×384 with 4 channels, and the two are stacked into **8 channels per frame**. We call DAM's own functions (`crop_image`, `get_image_tensor`, `encode_images_with_context`), so the shapes are the real ones.

In [ ]:
DEMO_ID = 1 if 1 in targets else targets[0]
fr = sampled[DEMO_ID]
imgs = [Image.fromarray(frames[f]) for f in fr]
msks = [Image.fromarray(traj.mask(DEMO_ID, f).astype(np.uint8) * 255) for f in fr]
panels, titles = [], []
for f, im, mk in zip(fr, imgs, msks):
    m = np.asarray(mk) > 0
    crop, info = dam_model.crop_image(im, mask_np=m.astype(np.uint8), crop_mode="focal_crop")   # DAM's own focal crop
    panels.append(overlay(frames[f], {DEMO_ID: m}, label=False)); titles.append(f"frame {f}: full view")
    c = np.asarray(crop).copy(); cm = info["mask_np"].astype(bool)
    panels.append(overlay(c, {DEMO_ID: cm}, label=False)); titles.append(f"focal crop {c.shape[1]}x{c.shape[0]} → 384x384")
show([cv2.resize(p, (240, 360)) for p in panels], titles, cols=8, size=2.2, save=f"6_dam_inputs_id{DEMO_ID}.png",
     suptitle=f"What DAM gets for object {DEMO_ID}: per frame a full view and a focal crop (coloured = mask)")

with torch.inference_mode():
    tens = [dam_model.get_image_tensor(im, mk, crop_mode="full", crop_mode2="focal_crop") for im, mk in zip(imgs, msks)]
    say(f"One frame's tensor: {tuple(tens[0].shape)} = (1 frame, 4 full-view channels + 4 focal channels, 384, 384)")
    try:
        feats = dam_model.model.encode_images_with_context(torch.cat(tens, 0))
        per_img, D = feats.shape[1], feats.shape[2]
        say(f"After SigLIP + cross-attention + projector: {tuple(feats.shape)} = (8 frames, {per_img} image tokens each, {D} numbers per token)")
    except Exception as e:
        per_img, D = None, None
        print("could not run encode_images_with_context directly:", repr(e))
from dam.model.mm_utils import tokenizer_image_token
from dam.model.constants import IMAGE_TOKEN_INDEX
ids_ = tokenizer_image_token(PROMPT, dam_model.tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt")
n_text = int((ids_ != IMAGE_TOKEN_INDEX).sum())
if per_img:
    say(f"Prompt = {n_text} text tokens + {n_img} × {per_img} image tokens = {n_text + n_img * per_img} positions in the language model")
display(HTML(highlight(PROMPT, fr, per_img)))

## 7. Run DAM on every selected object

This is exactly what `stage4_caption` (l.1558) does: for each target id, take its sampled frames and masks and call `captioner.describe`. For each object you'll see:
- its 8 frames (mask coloured);
- the conversation: the prompt, then **DAM's answer in purple**.

Generation uses `temperature 0.2`, `top_p 0.5`, `max_new_tokens 512`, with **no fixed seed**, just like their pipeline.

In [ ]:
described, t_all = [], time.time()
for k, oid in enumerate(targets):
    fr = sampled[oid]
    frame_imgs = [frames[f] for f in fr]
    masks = [traj.mask(oid, f) for f in fr]
    t0 = time.time()
    desc = captioner.describe(frame_imgs, masks)          # THEIR call
    dt = time.time() - t0
    described.append({"object_id": oid, "description": desc, "sampled_frames": fr,
                      "rank": rank_of[oid], "total_area": traj.total_area(oid), "seconds": round(dt, 1)})
    strip = [cv2.resize(overlay(frames[f], {oid: m}, label=False), (160, 280)) for f, m in zip(fr, masks)]
    fig, axes = plt.subplots(1, len(strip), figsize=(1.6 * len(strip), 3))
    for ax, im, f in zip(np.atleast_1d(axes), strip, fr):
        ax.imshow(im); ax.set_title(f"f{f}", fontsize=8); ax.axis("off")
    fig.suptitle(f"object {oid} (rank {rank_of[oid]}, visible in {len(traj._active[oid])} frames)", fontsize=10)
    fig.tight_layout(); fig.savefig(f"{OUT}/obj_{oid:02d}.png", dpi=50); plt.show()
    display(HTML(f"<b>object {oid}</b> · {dt:.1f} s" + highlight(PROMPT, fr, None, answer=desc)))
say(f"Described {len(described)} objects in {time.time() - t_all:.0f} s")

## 8. Is it reproducible? The same object, described 3 times

DAM samples words (temperature 0.2, top_p 0.5) and the pipeline sets no seed. Small differences here become different **attributes** in stage 5.

In [ ]:
oid = DEMO_ID
fr = sampled[oid]
runs = [next(d["description"] for d in described if d["object_id"] == oid)]
for _ in range(2):
    runs.append(captioner.describe([frames[f] for f in fr], [traj.mask(oid, f) for f in fr]))
for i, r_ in enumerate(runs):
    print(f"--- run {i + 1} ---\n{r_}\n")
same = sum(r_ == runs[0] for r_ in runs[1:])
w0 = runs[0].split()
for i, r_ in enumerate(runs[1:], 2):
    sm = difflib.SequenceMatcher(None, w0, r_.split())
    say(f"Reproducibility, object {oid}: run {i} vs run 1: identical={r_ == runs[0]}, word overlap {sm.ratio():.0%}")

## 9. Save

- `data/stage4/…_descriptions.json`: the stage-4 artifact in their format (`video`, `objects: [{object_id, description, sampled_frames}]`) plus `prompt`, rank, area and timing. **Push it with `git add -f`**, because `data/` is ignored. It's the input for stage 5.
- `notes/stage4/`: pictures + `README.md` with the prompt and every description.

In [ ]:
name = os.path.splitext(os.path.basename(VIDEO))[0]
art = {"video": VIDEO, "prompt": PROMPT, "caption_config": vars(cfg.caption), "objects": described}
path = f"{DATA}/{name}_first{T}_descriptions.json"
json.dump(art, open(path, "w"), indent=1)
lines = [f"# Stages 3–4 on `{name}` (first {T} frames)", "", *LOG, "", "## The prompt (identical for every object)", "", "```", PROMPT, "```", "",
         "![ranking](3_ranking.png)", "", "## Descriptions", ""]
for d in described:
    lines += [f"### object {d['object_id']} (rank {d['rank']}, frames {d['sampled_frames']})", "",
              f"![obj](obj_{d['object_id']:02d}.png)", "", "> " + d["description"].replace("\n", "\n> "), ""]
open(f"{OUT}/README.md", "w").write("\n".join(lines) + "\n")
print("saved", path, "and", OUT)
print("then in a terminal:\n  git add -f", path, "\n  git add notebooks/dam_step_by_step.ipynb", OUT,
      "\n  git commit -m 'stages 3-4 (DAM) run' && git push")

## 10. What to look for

1. **Section 2:** what did stage 3 delete? Was it really a duplicate or a part?
2. **Section 3:** which objects never get a name? Are some of them clearly real objects?
3. **Section 4:** how different are the code's frames and the paper's frames?
4. **Section 7:** do the paragraphs describe the right thing, or does a drifting mask make DAM describe a mix? Do the boy's and his shoe's paragraphs both talk about the shoe?
5. **Section 8:** how much does the text change between runs? Which words change (colours? actions?)